# Rakuten Image Classification - CNN Benchmark

**Clean, working version with all fixes applied**

## Key Fixes Applied (V1):
- ✅ Removed rescaling (EfficientNet expects [0, 255])
- ✅ Using float32 (no mixed precision issues)
- ✅ Memory-efficient subset creation using `.take()`
- ✅ Fixed hyperparameter references
- ✅ Proper model variable management

## Current Configuration:
- **Subset size:** 500 samples per class (~13,500 images)
- **Full Image Set** ~84000
- **Expected accuracy:** 62-68%
- **Training time:** ~60 minutes (EfficientNet) ~190 minutes (ConvNext)

## Changelog V2
- **Benchmark** cell 1: added benchmark configuration = MODELS_TO_BENCHMARK variable
- **New Model Build** cell 2: replaced model build part
- **Update Model Save** call 4: update model saving code line
- **New Cell 5** cell 5 (new): added for comparing results

## Changelog V3
- **Timer added** all cells: added timer code

##  Changelog V4
- **GPU Memory Error Fix** cell 1: removed .cacha() & fixed .take() code if images >1000

## Changelog V5
- **Fix for wrong batch validation** cell 1: removed .take() at all

## Changelog V6 (Kaggle Version 9)
- **Added ConvNext** cell 1 'models to benchmark' part & cell 2 Build Model section

## To Be Fixed
- **Tensorflow Error at End** cell 5 line 23: tensorflow call by model. call resulting in "Line AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" error at the end of run

---
## CELL 1: Setup & Data Loading

In [ ]:
"""
CELL 1: Setup, Imports, and Data Loading
"""

import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import classification_report, confusion_matrix
import gc

import time  # ← ADD THIS LINE
from datetime import timedelta

print("="*80)
print("RAKUTEN IMAGE CLASSIFICATION - EfficientNet")
print("="*80)
print(f"TensorFlow: {tf.__version__}")

# ============================================================================
# GPU SETUP - Use float32 (no mixed precision)
# ============================================================================

mixed_precision.set_global_policy('float32')  # Fixed: No mixed precision!
print("✅ Using float32 (no mixed precision)")

# Configure GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU ready: {len(gpus)} device(s)")
    except RuntimeError as e:
        print(f"GPU config error: {e}")
else:
    print("⚠️  No GPU detected - training will be VERY slow!")

AUTOTUNE = tf.data.AUTOTUNE

# ============================================================================
# CONFIGURATION
# ============================================================================

IMAGE_PATH = '/kaggle/input/rakuten-image-kaggle-dataset/images_organized'
OUTPUT_PATH = '/kaggle/working/'

# Subset settings
SAMPLES_PER_CLASS = 3158  # Adjust this: 200, 500, 1000, or 3158 (full)
BATCH_SIZE = 64
IMG_SIZE = 224

# Training settings
EPOCHS = 15
DROPOUT = 0.3
UNITS = 192
LR_PHASE1 = 0.001
LR_PHASE2 = 1e-5

# ============================================================================
# BENCHMARK CONFIGURATION
# ============================================================================

# Models to benchmark
MODELS_TO_BENCHMARK = [
    'EfficientNetB0',  # Already done: 56.9%
    'EfficientNetB1',  # Test this
    'ResNet50',        # Test this
    'ConvNeXtTiny'
]

CURRENT_MODEL = 'ConvNeXtTiny'  # ← CHANGE THIS for each run

print(f"\n🔬 BENCHMARK MODE")
print(f"   Current model: {CURRENT_MODEL}")
print(f"   Models to test: {', '.join(MODELS_TO_BENCHMARK)}")

print(f"\nConfiguration:")
print(f"  Samples/class: {SAMPLES_PER_CLASS}")
print(f"  Total images: ~{SAMPLES_PER_CLASS * 27}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Epochs: {EPOCHS}")

# ============================================================================
# LOAD DATASETS
# ============================================================================

print(f"\n{'='*80}")
print("LOADING DATASETS")
print(f"{'='*80}\n")

train_ds_full = tf.keras.utils.image_dataset_from_directory(
    IMAGE_PATH,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    seed=42,
    shuffle=True
)

val_ds_full = tf.keras.utils.image_dataset_from_directory(
    IMAGE_PATH,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=42,
    shuffle=True
)

class_names = train_ds_full.class_names
num_classes = len(class_names)

print(f"✅ Datasets loaded: {num_classes} classes\n")

# ============================================================================
# CREATE SUBSET (Memory-efficient using .take())
# ============================================================================

print(f"{'='*80}")
print(f"CREATING SUBSET: {SAMPLES_PER_CLASS} samples/class")
print(f"{'='*80}\n")

# Calculate how many batches to take
train_batches = (SAMPLES_PER_CLASS * num_classes) // BATCH_SIZE
val_batches = ((SAMPLES_PER_CLASS // 4) * num_classes) // BATCH_SIZE

print(f"Taking {train_batches} training batches (~{train_batches * BATCH_SIZE} images)")
print(f"Taking {val_batches} validation batches (~{val_batches * BATCH_SIZE} images)")

# CRITICAL FIX 1: Remove .cache() for full dataset!
#   Full or large dataset - NO caching (too much memory)
# CRITICAL FIX 2 : Conditional caching based on dataset size
if SAMPLES_PER_CLASS >= 1000:
    print("⚠️  Large dataset - caching DISABLED to prevent memory errors")
    train_ds = train_ds_full.prefetch(AUTOTUNE)  # ← Use ALL training data
    val_ds = val_ds_full.prefetch(AUTOTUNE)      # ← Use ALL validation data
else:
    # Small dataset - caching OK
    print("✅ Small dataset - caching enabled for speed")
    train_ds = train_ds_full.take(train_batches).cache().prefetch(AUTOTUNE)
    val_ds = val_ds_full.take(val_batches).cache().prefetch(AUTOTUNE)

print("\n✅ Subset created (memory-efficient)")
print(f"✅ Data loaded and ready for training\n")

---
## CELL 2: Build and Train Model

In [ ]:
"""
CELL 2: Model Building and Training
"""

print("="*80)
print("BUILDING MODEL")
print("="*80)

# ============================================================================
# BUILD MODEL - BENCHMARK VERSION
# ============================================================================

print(f"Building {CURRENT_MODEL}...")

# Start overall timer
training_start_time = time.time()
print(f"⏱️  Training started: {time.strftime('%H:%M:%S')}\n")

# Model selection
if CURRENT_MODEL == 'EfficientNetB0':
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        weights='imagenet',
        pooling='avg'
    )
    use_preprocessing = False  # EfficientNet: no preprocessing
    
elif CURRENT_MODEL == 'EfficientNetB1':
    base = tf.keras.applications.EfficientNetB1(
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        weights='imagenet',
        pooling='avg'
    )
    use_preprocessing = False  # EfficientNet: no preprocessing
    
elif CURRENT_MODEL == 'ResNet50':
    base = tf.keras.applications.ResNet50(
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        weights='imagenet',
        pooling='avg'
    )
    use_preprocessing = True  # ResNet: needs preprocessing

elif CURRENT_MODEL == 'ConvNeXtTiny':
    base = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        weights='imagenet',
        pooling='avg'
    )
    use_preprocessing = True  # ConvNeXt needs preprocessing
    
else:
    raise ValueError(f"Unknown model: {CURRENT_MODEL}")

base.trainable = False

# ============================================================================
# BUILD MODEL
# ============================================================================

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

if use_preprocessing:
    # Apply appropriate preprocessing based on model
    if 'ConvNeXt' in CURRENT_MODEL:
        x = tf.keras.applications.convnext.preprocess_input(inputs)
    elif 'ResNet' in CURRENT_MODEL:
        x = tf.keras.applications.resnet.preprocess_input(inputs)
    else:
        # Default preprocessing for other models
        x = inputs
    
    x = base(x, training=False)
else:
    # EfficientNet: no preprocessing (expects [0, 255])
    x = base(inputs, training=False)

# Add classification head
x = tf.keras.layers.Dropout(DROPOUT)(x)
x = tf.keras.layers.Dense(UNITS, activation='relu')(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_PHASE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"✅ {CURRENT_MODEL} built")
print(f"   Total params: {model.count_params():,}")

# Display preprocessing status
if use_preprocessing:
    if 'ConvNeXt' in CURRENT_MODEL:
        print(f"   Preprocessing: Yes (ConvNeXt)")
    elif 'ResNet' in CURRENT_MODEL:
        print(f"   Preprocessing: Yes (ResNet)")
else:
    print(f"   Preprocessing: No (EfficientNet)")

# ============================================================================
# PHASE 1: TRAIN WITH FROZEN BASE
# ============================================================================

print(f"\n{'='*80}")
print(f"PHASE 1: TRAINING WITH FROZEN BASE ({EPOCHS} EPOCHS)")
print(f"{'='*80}\n")

# Start Phase 1 timer
phase1_start = time.time()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f"\n{'='*80}")
print("PHASE 1 COMPLETE")
print(f"{'='*80}")
print(f"Final train accuracy: {history1.history['accuracy'][-1]:.4f} ({history1.history['accuracy'][-1]*100:.1f}%)")
print(f"Final val accuracy: {history1.history['val_accuracy'][-1]:.4f} ({history1.history['val_accuracy'][-1]*100:.1f}%)")

# Phase 1 time
phase1_time = time.time() - phase1_start
print(f"⏱️  Phase 1 time: {timedelta(seconds=int(phase1_time))}")

# ============================================================================
# PHASE 2: FINE-TUNING (only if Phase 1 worked well)
# ============================================================================

if history1.history['val_accuracy'][-1] > 0.30:
    print(f"\n{'='*80}")
    print(f"PHASE 2: FINE-TUNING (10 EPOCHS)")
    print(f"{'='*80}\n")
    
    # Start Phase 2 timer
    phase2_start = time.time()
    
    # Unfreeze base
    base.trainable = True
    
    # Freeze all except last 30 layers
    for layer in base.layers[:-30]:
        layer.trainable = False
    
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"Unfrozen {trainable} / {len(base.layers)} layers\n")
    
    # Recompile with lower LR
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_PHASE2),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=callbacks,
        verbose=1
    )
    
    print(f"\n{'='*80}")
    print("PHASE 2 COMPLETE")
    print(f"{'='*80}")
    print(f"Final train accuracy: {history2.history['accuracy'][-1]:.4f} ({history2.history['accuracy'][-1]*100:.1f}%)")
    print(f"Final val accuracy: {history2.history['val_accuracy'][-1]:.4f} ({history2.history['val_accuracy'][-1]*100:.1f}%)")
    
    # Phase 2 time
    phase2_time = time.time() - phase2_start
    print(f"⏱️  Phase 2 time: {timedelta(seconds=int(phase2_time))}")
    
else:
    # Phase 2 skipped
    phase2_time = 0
    print(f"\n⚠️  Skipping Phase 2 - Phase 1 accuracy too low")
    print(f"   Got {history1.history['val_accuracy'][-1]*100:.1f}%, need >30%")

---
## CELL 3: Evaluation and Analysis

In [ ]:
"""
CELL 3: Complete Model Evaluation
"""

print("="*80)
print("COMPLETE MODEL EVALUATION")
print("="*80)

# ============================================================================
# BASIC EVALUATION
# ============================================================================

print("\nEvaluating model...")
results = model.evaluate(val_ds, verbose=0)

print(f"\nValidation Loss: {results[0]:.4f}")
print(f"Validation Accuracy: {results[1]:.4f} ({results[1]*100:.1f}%)")

# ============================================================================
# DETAILED PREDICTIONS
# ============================================================================

print("\nGenerating predictions...")
y_pred = np.argmax(model.predict(val_ds, verbose=0), axis=1)
y_true = np.argmax(np.concatenate([y for _, y in val_ds]), axis=1)

# ============================================================================
# PER-CLASS PERFORMANCE
# ============================================================================

print(f"\n{'='*80}")
print("PER-CLASS PERFORMANCE")
print(f"{'='*80}")

report = classification_report(
    y_true, 
    y_pred, 
    target_names=[str(c) for c in class_names],
    digits=3
)
print(report)

# ============================================================================
# CONFUSION ANALYSIS
# ============================================================================

cm = confusion_matrix(y_true, y_pred)
print(f"\n{'='*80}")
print("MOST CONFUSED CLASS PAIRS")
print(f"{'='*80}")

confusion_pairs = []
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append((cm[i, j], class_names[i], class_names[j]))

confusion_pairs.sort(reverse=True)
for count, true_class, pred_class in confusion_pairs[:10]:
    print(f"  True: {true_class:>4} → Predicted: {pred_class:>4} ({count:>3} times)")

# ============================================================================
# OVERFITTING CHECK
# ============================================================================

train_results = model.evaluate(train_ds, verbose=0)
gap = train_results[1] - results[1]

print(f"\n{'='*80}")
print("OVERFITTING CHECK")
print(f"{'='*80}")
print(f"Train Accuracy: {train_results[1]:.4f} ({train_results[1]*100:.1f}%)")
print(f"Val Accuracy:   {results[1]:.4f} ({results[1]*100:.1f}%)")
print(f"Gap:            {gap:.4f} ({gap*100:.1f}%)")

if gap > 0.15:
    print("⚠️  Significant overfitting (>15% gap)")
    print("   → More data will help reduce this")
elif gap > 0.08:
    print("✅ Moderate overfitting (8-15% gap) - normal for subsets")
else:
    print("✅ Good generalization (<8% gap)")

# ============================================================================
# DIVERSITY CHECK
# ============================================================================

unique_predictions = len(set(y_pred))
print(f"\nPrediction diversity: {unique_predictions}/{num_classes} classes predicted")

# ============================================================================
# COMPARISON WITH PREVIOUS RESULTS
# ============================================================================

print(f"\n{'='*80}")
print("SUMMARY & COMPARISON")
print(f"{'='*80}")
print(f"\nCurrent results ({SAMPLES_PER_CLASS} samples/class):")
print(f"  Validation Accuracy: {results[1]*100:.1f}%")
print(f"  Overfitting gap: {gap*100:.1f}%")
print(f"  Classes predicted: {unique_predictions}/{num_classes}")

print(f"\n💡 INTERPRETATION:")
if results[1] >= 0.65:
    print("  🎉 EXCELLENT! (≥65%)")
    print("     → Ready for full dataset training")
    print("     → Expected full dataset: 76-82%")
elif results[1] >= 0.58:
    print("  ✅ GOOD! (58-65%)")
    print("     → Solid improvement from 51%")
    print("     → Can proceed to full dataset")
elif results[1] >= 0.52:
    print("  ✅ MODEST IMPROVEMENT (52-58%)")
    print("     → More data is helping")
    print("     → Try 1,000 samples/class next")
else:
    print(f"  ⚠️  SIMILAR TO BASELINE (<52%)")
    print("     → Check if anything went wrong")

print(f"\n✅ Evaluation complete!")

# ============================================================================
# TIMING SUMMARY
# ============================================================================

total_time = time.time() - training_start_time

print(f"\n{'='*80}")
print("TIMING SUMMARY")
print(f"{'='*80}")
print(f"Phase 1 (frozen base):  {timedelta(seconds=int(phase1_time))}")
if phase2_time > 0:
    print(f"Phase 2 (fine-tuning):  {timedelta(seconds=int(phase2_time))}")
    print(f"Training total:         {timedelta(seconds=int(phase1_time + phase2_time))}")
else:
    print(f"Phase 2 (fine-tuning):  SKIPPED")
    print(f"Training total:         {timedelta(seconds=int(phase1_time))}")
print(f"Evaluation + overhead:  {timedelta(seconds=int(total_time - phase1_time - phase2_time))}")
print(f"Total time:             {timedelta(seconds=int(total_time))}")
print(f"\n⏱️  Average: {results[1]/total_time*100:.2f}% accuracy per minute")
print(f"⏱️  Efficiency: {results[1]/(total_time/60):.4f} acc/min")

---
## CELL 4: Save Model

In [ ]:
"""
CELL 4: Save Model and Export Results
"""

# Save model
model_name = f'{CURRENT_MODEL.lower()}_{SAMPLES_PER_CLASS}samples.keras'
model.save(f'/kaggle/working/{model_name}')
print(f"✅ Model saved: {model_name}")

# Save predictions for reproducibility
np.save('/kaggle/working/val_predictions.npy', model.predict(val_ds, verbose=0))
np.save('/kaggle/working/val_true_labels.npy', np.concatenate([y for _, y in val_ds]))
print(f"✅ Predictions saved for reproducibility")

print(f"\n{'='*80}")
print("NEXT STEPS")
print(f"{'='*80}")
print(f"\n1. Download the model: {model_name}")
print(f"2. Share with teammates for evaluation")
print(f"3. If satisfied with results:")
if SAMPLES_PER_CLASS < 1000:
    print(f"   → Try 1,000 samples/class (expected: 68-73%)")
if SAMPLES_PER_CLASS < 3000:
    print(f"   → Or jump to full dataset (expected: 76-82%)")
print(f"4. Consider multimodal approach (text + images)")
print(f"\n✅ ALL DONE!")

---
## CELL 5: Benchmark Results Tracker

In [ ]:
"""
CELL 5 (NEW): Benchmark Results Tracker - SIMPLIFIED
"""

import json
import os

benchmark_file = '/kaggle/working/benchmark_results.json'

# Load existing results if file exists
if os.path.exists(benchmark_file):
    with open(benchmark_file, 'r') as f:
        benchmark_results = json.load(f)
else:
    benchmark_results = {}

# Save current model results (with all fields from Cell 3)
benchmark_results[CURRENT_MODEL] = {
    'samples_per_class': SAMPLES_PER_CLASS,
    'val_accuracy': float(results[1]),
    'train_accuracy': float(train_results[1]),
    'overfitting_gap': float(gap),
    'parameters': int(model.count_params()),
    'epochs_trained': len(history1.history['loss']),
    'unique_predictions': int(unique_predictions),
    'model_file': model_name
}

# Add timing if available (Cell 3 creates these variables)
if 'total_time' in globals():
    benchmark_results[CURRENT_MODEL]['total_time_minutes'] = float(total_time / 60)
    benchmark_results[CURRENT_MODEL]['phase1_time_minutes'] = float(phase1_time / 60)
    benchmark_results[CURRENT_MODEL]['phase2_time_minutes'] = float(phase2_time / 60)
    benchmark_results[CURRENT_MODEL]['efficiency_acc_per_min'] = float(results[1] / (total_time / 60))

# Save to file
with open(benchmark_file, 'w') as f:
    json.dump(benchmark_results, f, indent=2)

print(f"✅ Results saved for {CURRENT_MODEL}")

# ============================================================================
# DISPLAY ALL RESULTS
# ============================================================================

print(f"\n{'='*80}")
print("BENCHMARK RESULTS SUMMARY")
print(f"{'='*80}\n")

if len(benchmark_results) > 0:
    # Simple display - always works
    print(f"{'Model':<20} {'Val Acc':<10} {'Train Acc':<10} {'Gap':<8} {'Params':<10}")
    print("-" * 70)
    
    for model_name_key, res in sorted(benchmark_results.items()):
        print(f"{model_name_key:<20} "
              f"{res['val_accuracy']*100:>6.1f}%   "
              f"{res['train_accuracy']*100:>6.1f}%    "
              f"{res['overfitting_gap']*100:>5.1f}%  "
              f"{res['parameters']/1e6:>6.1f}M")
    
    # Find best model
    best_model = max(benchmark_results.items(), key=lambda x: x[1]['val_accuracy'])
    print(f"\n🏆 BEST MODEL: {best_model[0]} ({best_model[1]['val_accuracy']*100:.1f}%)")
    
    # Show timing if available
    print(f"\n📊 TIMING INFO (if available):")
    for model_name_key, res in sorted(benchmark_results.items()):
        if 'total_time_minutes' in res:
            print(f"   {model_name_key}: {res['total_time_minutes']:.1f} min "
                  f"(efficiency: {res['efficiency_acc_per_min']:.3f} acc/min)")
        else:
            print(f"   {model_name_key}: timing data not available")
else:
    print("No results yet. Run models first!")

print(f"\n{'='*80}")
print("NEXT STEPS")
print(f"{'='*80}")

tested = set(benchmark_results.keys())
remaining = set(MODELS_TO_BENCHMARK) - tested

if remaining:
    print(f"\n📝 Models remaining to test:")
    for m in remaining:
        print(f"   → {m}")
    print(f"\n🔧 TO TEST NEXT MODEL:")
    print(f"   1. Go to CELL 1")
    print(f"   2. Change: CURRENT_MODEL = '{list(remaining)[0]}'")
    print(f"   3. Run: Kernel → Restart & Run All")
else:
    print(f"\n✅ All models tested!")
    print(f"   → Review results above")
    print(f"   → Choose best model for scaling to full dataset")